In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"  

!pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer -q
!pip install -U unsloth trl --break-system-packages -q
!pip install -q transformers==4.56.2 -q

In [ ]:
import os
import random
import argparse
import numpy as np
import torch

from datasets import Dataset, Image as HFImage, concatenate_datasets
from torchvision import transforms as T
from PIL import Image

from unsloth import FastVisionModel
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f"Device : {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"BF16   : {torch.cuda.is_bf16_supported()}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import wandb
import os
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("hugging_face")
    os.environ["HF_TOKEN"] = secret_value_0
    
    secret_value_1 = user_secrets.get_secret("wandb.login")
    os.environ["WANDB_API_KEY"] = secret_value_1  
    !wandb login {os.environ["WANDB_API_KEY"]} 
    
except Exception as e:
    print('API Key not found', str(e))

In [ ]:
# Dataset path 
MVTEC_PATH = "/kaggle/input/datasets/ipythonx/mvtec-ad"          
VISA_PATH  = "/kaggle/input/datasets/ess1004/visa-anomaly-detection"              

# Model 
MODEL_NAME     = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024

# LoRA 
LORA_R       = 16
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05

# Training 
EPOCHS       = 1
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
LR           = 2e-4
WARMUP_RATIO = 0.05
TEST_SIZE    = 0.1      
SAVE_STEPS   = 200
LOG_STEPS    = 10
SEED         = 42

OUTPUT_DIR   = "/kaggle/working/qwen25vl_mvtec_visa"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# W&B 
WANDB_PROJECT  = "qwen25vl-anomaly-detection"
WANDB_RUN_NAME = "mvtec-visa-qlora" 

In [ ]:
from huggingface_hub import create_repo

create_repo(
    repo_id="songthienll/qwen25vl-mvtec-visa-checkpoint",
    repo_type="model",
    private=True, 
    token=os.environ["HF_TOKEN"],
    exist_ok=True,  
)
print("Repo created")

In [ ]:
# Dataset loaders

def load_mvtec_dataset(base_dir):
    data = []
    for category in sorted(os.listdir(base_dir)):
        cat_path = os.path.join(base_dir, category)
        if not os.path.isdir(cat_path):
            continue

        # Train – normal
        train_good = os.path.join(cat_path, "train", "good")
        if os.path.exists(train_good):
            for fname in os.listdir(train_good):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    data.append({
                        "image"     : os.path.join(train_good, fname),
                        "label"     : "no anomaly",
                        "class_name": category,
                        "source"    : "mvtec",
                    })

        # Test – good + anomaly types
        test_path = os.path.join(cat_path, "test")
        if os.path.exists(test_path):
            for defect in os.listdir(test_path):
                defect_path = os.path.join(test_path, defect)
                if not os.path.isdir(defect_path):
                    continue
                label = "no anomaly" if defect == "good" else defect.replace("_", " ")
                for fname in os.listdir(defect_path):
                    if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                        data.append({
                            "image"     : os.path.join(defect_path, fname),
                            "label"     : label,
                            "class_name": category,
                            "source"    : "mvtec",
                        })

    cats = len(set(d["class_name"] for d in data))
    print(f"[MVTec-AD]  {len(data):,} ảnh | {cats} categories")
    ds = Dataset.from_list(data)
    ds = ds.cast_column("image", HFImage())
    return ds

mvtec_ds = load_mvtec_dataset(MVTEC_PATH) if MVTEC_PATH else None

def load_visa_dataset(base_dir):
    data = []
    categories_found = set()

    # Layout B: CSV splits (official VisA release)
    csv_root = os.path.join(base_dir, "split_csv", "1cls")
    if os.path.isdir(csv_root):
        for csv_file in sorted(os.listdir(csv_root)):
            if not csv_file.endswith(".csv"):
                continue
            category = csv_file[:-4]
            categories_found.add(category)
            with open(os.path.join(csv_root, csv_file), newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    img_path = os.path.join(base_dir, row["image"])
                    if not os.path.exists(img_path):
                        continue
                    raw = row.get("label", "").strip().lower()
                    label = "no anomaly" if raw in ("normal", "good", "0") else (raw or "anomaly")
                    data.append({
                        "image"     : img_path,
                        "label"     : label,
                        "class_name": category,
                        "source"    : "visa",
                    })

    # Layout A: Normal / Anomaly sub-folders
    if not data:
        for category in sorted(os.listdir(base_dir)):
            cat_path = os.path.join(base_dir, category)
            if not os.path.isdir(cat_path):
                continue
            for folder, label in [
                (os.path.join(cat_path, "Data", "Images", "Normal"),  "no anomaly"),
                (os.path.join(cat_path, "Data", "Images", "Anomaly"), "anomaly"),
            ]:
                if not os.path.exists(folder):
                    continue
                categories_found.add(category)
                for fname in os.listdir(folder):
                    if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                        data.append({
                            "image"     : os.path.join(folder, fname),
                            "label"     : label,
                            "class_name": category,
                            "source"    : "visa",
                        })

    if not data:
        raise RuntimeError(
            f"Không load được VisA từ '{base_dir}'.\n"
            "Kiểm tra path và cấu trúc thư mục (Normal/Anomaly hoặc split_csv)."
        )

    cats = len(categories_found)
    print(f"[VisA]      {len(data):,} ảnh | {cats} categories")
    ds = Dataset.from_list(data)
    ds = ds.cast_column("image", HFImage())
    return ds

visa_ds = load_visa_dataset(VISA_PATH) if VISA_PATH else None

In [ ]:
from datasets import concatenate_datasets

parts = [ds for ds in [mvtec_ds, visa_ds] if ds is not None]

combined = concatenate_datasets(parts).shuffle(seed=SEED)
print(f"[Combined]  {len(combined):,} ảnh tổng cộng")

# Thống kê label
from collections import Counter
label_counts = Counter(combined["label"])
print(f"  no anomaly : {label_counts.get('no anomaly', 0):,}")
print(f"  anomaly    : {sum(v for k,v in label_counts.items() if k != 'no anomaly'):,}")

split = combined.train_test_split(test_size=TEST_SIZE, seed=SEED)
train_raw, test_raw = split["train"], split["test"]
print(f"Train: {len(train_raw):,} | Eval: {len(test_raw):,}")


In [ ]:
SYSTEM_PROMPT = (
    "You are an expert industrial quality-control inspector. "
    "Analyze the provided image and detect any surface defects or anomalies."
)

def convert_to_conversation(sample):
    class_name = sample["class_name"]
    label      = sample["label"]

    user_text = (
        f"Analyze this {class_name} for quality control.\n"
        "Briefly describe:\n"
        "- The object's overall condition\n"
        "- Whether any anomaly or defect is present\n"
        "- The type and location of the defect (if any)\n"
        "Answer in 2-3 concise sentences."
    )

    if label == "no anomaly":
        assistant_text = (
            f"The {class_name} appears to be in perfect condition with no visible defects. "
            "The surface is uniform and free from any anomalies. "
            "This is a normal, defect-free sample."
        )
    else:
        assistant_text = (
            f"The {class_name} shows a defect classified as '{label}'. "
            f"An anomaly is visible on the surface of the {class_name}. "
            "This sample requires further inspection or rejection."
        )

    return {
        "messages": [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}],
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": sample["image"]},
                    {"type": "text",  "text": user_text},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": assistant_text}],
            },
        ]
    }

train_ds = train_raw.map(convert_to_conversation, remove_columns=train_raw.column_names)
test_ds  = test_raw.map(convert_to_conversation,  remove_columns=test_raw.column_names)
print(f" Train: {len(train_ds):,} | Eval: {len(test_ds):,}")

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True, #QLora
    use_gradient_checkpointing="unsloth",
)


# Load lại checkpoint đã push lên Hub
# model, tokenizer = FastVisionModel.from_pretrained(
#     "songthienll/qwen25vl-mvtec-visa-checkpoint",  
#     load_in_4bit=True,
#     use_gradient_checkpointing="unsloth",
#     token=os.environ["HF_TOKEN"],
# )

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    random_state=SEED,
    use_rslora=False,
)
model.print_trainable_parameters()


In [ ]:
# Tổng số params trainable
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())

run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        # Model
        "model_name"      : MODEL_NAME,
        "max_seq_length"  : MAX_SEQ_LENGTH,
        # LoRA
        "lora_r"          : LORA_R,
        "lora_alpha"      : LORA_ALPHA,
        "lora_dropout"    : LORA_DROPOUT,
        # Training
        "epochs"          : EPOCHS,
        "batch_size"      : BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM,
        "effective_batch" : BATCH_SIZE * GRAD_ACCUM,
        "learning_rate"   : LR,
        "warmup_ratio"    : WARMUP_RATIO,
        "lr_scheduler"    : "cosine",
        "optimizer"       : "adamw_8bit",
        "seed"            : SEED,
        # Params
        "trainable_params": trainable_params,
        "total_params"    : total_params,
        "trainable_pct"   : round(100 * trainable_params / total_params, 2),
    },
)

print(f"Wandb's URL: {run.url}")
print(f"Project : {WANDB_PROJECT}")
print(f"Run     : {run.name}")

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator
from transformers import TrainerCallback

FastVisionModel.for_training(model)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    # Eval
    eval_strategy="steps",
    eval_steps=100,
    # Save
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logging
    logging_steps=LOG_STEPS,
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    report_to="wandb",    
    run_name=WANDB_RUN_NAME,
    remove_unused_columns=False,
    seed=SEED,
)

class PushToHubCallback(TrainerCallback):
    def __init__(self, model, tokenizer, repo_id, token):
        self.model     = model
        self.tokenizer = tokenizer
        self.repo_id   = repo_id
        self.token     = token

    def on_save(self, args, state, control, **kwargs):
        from huggingface_hub import HfApi
        api = HfApi()
        api.upload_folder(
            folder_path=f"{args.output_dir}/checkpoint-{state.global_step}",
            repo_id=self.repo_id,
            repo_type="model",
            token=self.token,
            commit_message=f"checkpoint step {state.global_step}",
        )
        print(f" Pushed full checkpoint step {state.global_step}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    callbacks=[PushToHubCallback(model, tokenizer, repo_id="songthienll/qwen25vl-mvtec-visa-checkpoint", token=os.environ["HF_TOKEN"])]
)

In [ ]:
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free/1e9:.1f} / {total/1e9:.1f} GB")
    wandb.log({"system/vram_free_gb": free/1e9, "system/vram_total_gb": total/1e9}, step=0)

trainer_stats = trainer.train()

# from huggingface_hub import snapshot_download
# ckpt_path = snapshot_download(
#     repo_id="songthienll/qwen25vl-mvtec-visa-checkpoint",
#     token=os.environ["HF_TOKEN"],
# )
# trainer_stats = trainer.train(resume_from_checkpoint=ckpt_path)

# Summary
train_loss    = trainer_stats.training_loss
runtime_min   = trainer_stats.metrics.get("train_runtime", 0) / 60
samples_per_s = trainer_stats.metrics.get("train_samples_per_second", 0)

print(f"\n{'='*50}")
print(f"   Training completed")
print(f"   Total steps      : {trainer_stats.global_step}")
print(f"   Training loss    : {train_loss:.4f}")
print(f"   Time elapsed     : {runtime_min:.1f} phút")
print(f"   Samples/sec      : {samples_per_s:.2f}")

# Log final summary to W&B
wandb.summary["train/final_loss"]       = train_loss
wandb.summary["train/total_steps"]      = trainer_stats.global_step
wandb.summary["train/runtime_minutes"]  = round(runtime_min, 2)
wandb.summary["train/samples_per_sec"]  = round(samples_per_s, 2)

In [ ]:
final_dir = os.path.join(OUTPUT_DIR, "final")
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f" Model saved at: {final_dir}")

total_size = 0
for f in os.listdir(final_dir):
    size = os.path.getsize(os.path.join(final_dir, f))
    total_size += size
    print(f"  {f:<45} {size/1e6:>8.1f} MB")
print(f"  {'TOTAL':<45} {total_size/1e6:>8.1f} MB")

# Upload model artifact to W&B 
artifact = wandb.Artifact(
    name=f"qwen25vl-mvtec-visa-lora",
    type="model",
    description="Qwen2.5-VL 7B QLoRA fine-tuned on MVTec-AD + VisA",
    metadata={
        "model_name"  : MODEL_NAME,
        "lora_r"      : LORA_R,
        "lora_alpha"  : LORA_ALPHA,
        "epochs"      : EPOCHS,
        "train_loss"  : round(train_loss, 4),
        "train_images": DATASET_STATS["dataset/train_images"],
    },
)
artifact.add_dir(final_dir, name="lora_weights")
wandb.log_artifact(artifact)
print("Model artifact uploaded to W&B")

In [ ]:
from PIL import Image as PILImage
import numpy as np

FastVisionModel.for_inference(model)

def predict(image, class_name="object"):
    """Nhận PIL Image, trả về chuỗi mô tả."""
    messages = [
        {"role": "system",    "content": [{"type": "text",  "text": SYSTEM_PROMPT}]},
        {"role": "user",      "content": [{"type": "image", "image": image},
                                          {"type": "text",  "text": (
                                              f"Analyze this {class_name} for quality control.\n"
                                              "Describe: overall condition, anomaly, defect type and location."
                                          )}]},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[text], images=[image], return_tensors="pt").to("cuda")
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=False)
    return tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

N_SAMPLES = 20   

table = wandb.Table(columns=["image", "category", "source", "ground_truth", "prediction"])

print(f"Đang chạy inference trên {N_SAMPLES} mẫu eval...")
for i in range(min(N_SAMPLES, len(test_raw))):
    sample = test_raw[i]

    raw_img = sample["image"]
    if isinstance(raw_img, dict) and "path" in raw_img:
        pil_img = PILImage.open(raw_img["path"])
    else:
        pil_img = raw_img

    prediction = predict(pil_img, sample["class_name"])

    # Resize to upload to W&B
    thumb = pil_img.convert("RGB")
    thumb.thumbnail((256, 256))

    table.add_data(
        wandb.Image(thumb),
        sample["class_name"],
        sample["source"],
        sample["label"],
        prediction,
    )
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{N_SAMPLES} done")

wandb.log({"eval/prediction_samples": table})

In [ ]:
wandb.finish()
print(f" Results: https://wandb.ai/{wandb.Api().default_entity}/{WANDB_PROJECT}")

In [ ]:
# Save model
!huggingface-cli login --token $HF_TOKEN
from huggingface_hub import notebook_login
notebook_login()

model.push_to_hub("songthienll/qwen2.5-vl-7b-mvtec-visa")
tokenizer.push_to_hub("songthienll/qwen2.5-vl-7b-mvtec-visa", token=True)